In [ ]:
import random
import numpy as np
import pandas as pd
import torch

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

TRAIN_PATH = "train.csv"

train_df = pd.read_csv(TRAIN_PATH)
train_df = train_df[["prompt", "svg"]].dropna().reset_index(drop=True)

def good_svg(svg):
    if not isinstance(svg, str):
        return False

    l = len(svg)
    p = svg.count("<path")
    c = svg.count("<circle")
    r = svg.count("<rect")
    e = svg.count("<ellipse")
    g = svg.count("<polygon")
    line = svg.count("<line")

    if l < 180:
        return False

    if l > 4500:
        return False

    if p > 25:
        return False

    if "<script" in svg or "<foreignObject" in svg:
        return False

    shape_total = p + c + r + e + g + line

    if shape_total <= 1:
        return False

    if p == 1 and c == 0 and r == 0 and e == 0 and g == 0 and line == 0:
        return False

    return True

train_df = train_df[train_df["svg"].apply(good_svg)].reset_index(drop=True)

sample_n = min(12000, len(train_df))
train_df = train_df.sample(sample_n, random_state=42).reset_index(drop=True)

def build_text(prompt, svg):
    return f"Text: {prompt}\nSVG: {svg}"

train_df["text"] = train_df.apply(lambda x: build_text(x["prompt"], x["svg"]), axis=1)

print("filtered train size:", len(train_df))
train_df.head()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments

OUTPUT_DIR = "./lora_svg_run"
SAVE_PATH = "./lora_svg_textsvg_final"

dataset = Dataset.from_pandas(train_df[["text"]])

def tokenize(x):
    out = tokenizer(
        x["text"],
        truncation=True,
        max_length=768,
        padding="max_length"
    )
    out["labels"] = out["input_ids"].copy()
    return out

dataset = dataset.map(tokenize, batched=False)
dataset = dataset.remove_columns(["text"])

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=0.8,
    learning_rate=1e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("saved to:", SAVE_PATH)